# Novo Nordisk Part 2 Step A: Signal Computation

**Author:** Roshan Patel  
**Course:** 15.465 Alphanomics, Spring 2026  
**Cutoff date:** 30 April 2026  
**Reporting basis:** DKK for income-statement signals (IFRS native reporting). USD ADR for Momentum and Volatility.

This notebook produces every value in the Step A signal table. Each section is independently runnable. Data caches live in `data/` so re-running does not re-fetch.

Dependencies: `pandas`, `numpy`, `requests`, `yfinance`. WRDS endpoints require the `WRDS_API_TOKEN` environment variable to be set.

## Setup

In [1]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import yfinance as yf

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

WRDS_TOKEN = os.environ.get('WRDS_API_TOKEN')
if not WRDS_TOKEN:
    raise RuntimeError(
        'Set WRDS_API_TOKEN before running. In a shell: '
        'export WRDS_API_TOKEN=your_token_here'
    )

WRDS_BASE = 'https://wrds-api.wharton.upenn.edu/data'

/Users/Roshan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. Reference dates

The team is using end of April 2026 as the project cutoff. NVO Q4 2025 results were released on 3 February 2026 (Bagsværd, Denmark, 11:38 ET). Q1 2026 results are scheduled for 6 May 2026, after the cutoff, so the most recent quarterly print available for Earnings Surprise and SUE is Q4 2025.

In [2]:
CUTOFF = pd.Timestamp('2026-04-30')
Q4_2025_ANNOUNCEMENT = pd.Timestamp('2026-02-03')
MOMENTUM_WINDOW_END = pd.Timestamp('2026-03-31')   # skip most recent month
MOMENTUM_WINDOW_START = pd.Timestamp('2025-03-31')
VOL_WINDOW_END = CUTOFF
VOL_WINDOW_START = CUTOFF - pd.Timedelta(days=365)
INSIDER_WINDOW_START = CUTOFF - pd.DateOffset(months=3)

print('Project cutoff:           ', CUTOFF.date())
print('Q4 2025 announcement:     ', Q4_2025_ANNOUNCEMENT.date())
print('Momentum window:          ', MOMENTUM_WINDOW_START.date(), '->', MOMENTUM_WINDOW_END.date())
print('Volatility window:        ', VOL_WINDOW_START.date(), '->', VOL_WINDOW_END.date())
print('Insider 3-month window:   ', INSIDER_WINDOW_START.date(), '->', CUTOFF.date())

Project cutoff:            2026-04-30
Q4 2025 announcement:      2026-02-03
Momentum window:           2025-03-31 -> 2026-03-31
Volatility window:         2025-04-30 -> 2026-04-30
Insider 3-month window:    2026-01-30 -> 2026-04-30


## 2. WRDS IBES data

IBES is the source for analyst consensus and reported quarterly EPS, in DKK as NVO reports them. Two pulls:

1. Quarterly EPS actuals through Q3 2025 (`ibes.actu_epsus`).
2. FY 2025 annual consensus history (`ibes.statsum_epsus`); the most recent statpers before the 3 Feb 2026 announcement is the relevant snapshot.

WRDS had not yet ingested NVO Q4 2025 actual at the time of pulling, so Q4 2025 EPS is recovered as `FY 2025 reported (press release) − sum(Q1-Q3 2025 IBES actuals)`. Same logic produces the implied Q4 2025 consensus from the latest annual consensus.

In [3]:
def wrds_get(path, params, cache_path=None):
    """Hit a WRDS REST endpoint and return the parsed JSON. Caches to disk if cache_path given."""
    if cache_path and cache_path.exists():
        return json.loads(cache_path.read_text())
    headers = {
        'Accept': 'application/json',
        'Authorization': f'Token {WRDS_TOKEN}',
    }
    r = requests.get(f'{WRDS_BASE}/{path}/', headers=headers, params=params, timeout=30)
    r.raise_for_status()
    payload = r.json()
    if cache_path:
        cache_path.write_text(json.dumps(payload, indent=2))
    return payload

# 2.1 Quarterly EPS actuals (DKK), 2024 onward
actuals = wrds_get(
    'ibes.actu_epsus',
    params={'ticker': 'NVO', 'pdicity': 'QTR', 'pends__gte': '2024-01-01'},
    cache_path=DATA_DIR / 'ibes_nvo_quarterly_actuals.json',
)
actu_df = pd.DataFrame(actuals['results'])[['pends', 'anndats', 'pdicity', 'value', 'curr_act']]
# WRDS returns both ANN and QTR rows even when pdicity=QTR is requested. Force-filter.
actu_df = actu_df[actu_df['pdicity'] == 'QTR'].sort_values('pends').reset_index(drop=True)
print(actu_df)

        pends     anndats pdicity  value curr_act
0  2024-03-31  2024-05-02     QTR   5.68      DKK
1  2024-06-30  2024-08-07     QTR   4.49      DKK
2  2024-09-30  2024-11-06     QTR   6.12      DKK
3  2024-12-31  2025-02-05     QTR   6.34      DKK
4  2025-03-31  2025-05-07     QTR   6.53      DKK
5  2025-06-30  2025-08-06     QTR   5.96      DKK
6  2025-09-30  2025-11-05     QTR   4.50      DKK


In [4]:
# 2.2 FY 2025 annual consensus history
consensus = wrds_get(
    'ibes.statsum_epsus',
    params={'ticker': 'NVO', 'fpedats': '2025-12-31', 'measure': 'EPS', 'fiscalp': 'ANN', 'limit': 50},
    cache_path=DATA_DIR / 'ibes_nvo_fy2025_annual_consensus.json',
)
cons_df = pd.DataFrame(consensus['results'])[['statpers', 'meanest', 'medest', 'numest', 'stdev', 'curcode']]
cons_df['statpers'] = pd.to_datetime(cons_df['statpers'])
cons_df = cons_df.sort_values('statpers').reset_index(drop=True)
print(cons_df.tail(10))

     statpers  meanest  medest  numest  stdev curcode
40 2025-05-15    26.53   26.23    19.0   0.95     DKK
41 2025-06-19    26.20   26.15    20.0   0.94     DKK
42 2025-07-17    26.09   25.98    20.0   0.99     DKK
43 2025-08-14    24.92   24.77    16.0   0.76     DKK
44 2025-09-18    24.10   24.33    17.0   0.76     DKK
45 2025-10-16    23.62   23.47    18.0   0.87     DKK
46 2025-11-20    23.36   23.02    17.0   0.79     DKK
47 2025-12-18    23.27   22.93    18.0   0.82     DKK
48 2026-01-15    23.33   22.97    19.0   0.81     DKK
49 2026-02-19    23.22   22.92    20.0   0.81     DKK


In [5]:
# 2.3 Most recent statpers before the 3 Feb 2026 announcement
latest_consensus_row = cons_df[cons_df['statpers'] < Q4_2025_ANNOUNCEMENT].iloc[-1]
fy_2025_consensus_dkk = float(latest_consensus_row['meanest'])
print(f"Latest pre-announcement annual consensus: {fy_2025_consensus_dkk:.2f} DKK "
      f"(statpers={latest_consensus_row['statpers'].date()}, n_est={int(latest_consensus_row['numest'])})")

# Q1-Q3 2025 actuals (DKK) sum
q123_2025 = actu_df[actu_df['pends'].isin(['2025-03-31', '2025-06-30', '2025-09-30'])]
q123_sum = float(q123_2025['value'].astype(float).sum())
print(f"Q1+Q2+Q3 2025 actuals (DKK): {q123_sum:.2f}")

# FY 2025 reported (from NVO Q4 2025 press release dated 3 Feb 2026)
FY_2025_REPORTED_DKK = 23.03

# Imputed Q4 2025 actual and Q4 2025 implied consensus
Q4_2025_ACTUAL_DKK = FY_2025_REPORTED_DKK - q123_sum
Q4_2025_CONSENSUS_DKK = fy_2025_consensus_dkk - q123_sum
Q4_2024_ACTUAL_DKK = float(actu_df.loc[actu_df['pends'] == '2024-12-31', 'value'].iloc[0])

print(f"Q4 2025 actual (imputed):     {Q4_2025_ACTUAL_DKK:.4f} DKK")
print(f"Q4 2025 consensus (imputed):  {Q4_2025_CONSENSUS_DKK:.4f} DKK")
print(f"Q4 2024 actual (IBES):        {Q4_2024_ACTUAL_DKK:.4f} DKK")

Latest pre-announcement annual consensus: 23.33 DKK (statpers=2026-01-15, n_est=19)
Q1+Q2+Q3 2025 actuals (DKK): 16.99
Q4 2025 actual (imputed):     6.0400 DKK
Q4 2025 consensus (imputed):  6.3400 DKK
Q4 2024 actual (IBES):        6.3400 DKK


## 3. Yahoo! Finance prices

Three series: NVO ADR (USD, NYSE), S&P 500 (USD, market benchmark for Momentum), and NOVO-B (DKK, Copenhagen, used for the price-5-trading-days-prior in the Earnings Surprise / SUE denominator since the EPS values are in DKK).

Window covers April 2024 through April 2026 to support the 12-month Volatility window (ending 30 Apr 2026) and the 12-month Momentum window (ending 31 Mar 2026).

In [6]:
def get_prices(ticker, start='2024-04-01', end='2026-05-01'):
    fname = DATA_DIR / f"prices_{ticker.replace('^', '').replace('.', '_')}.csv"
    if fname.exists():
        return pd.read_csv(fname, parse_dates=['Date'], index_col='Date')
    df = yf.download(ticker, start=start, end=end, auto_adjust=False, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    df.to_csv(fname)
    return df

nvo = get_prices('NVO')
spx = get_prices('^GSPC')
novob = get_prices('NOVO-B.CO')

print('NVO ADR (USD)         last close:', nvo["Adj Close"].iloc[-1])
print('S&P 500               last close:', spx["Adj Close"].iloc[-1])
print('NOVO-B Copenhagen DKK last close:', novob["Adj Close"].iloc[-1])

NVO ADR (USD)         last close: 42.220001220703125
S&P 500               last close: 7209.009765625
NOVO-B Copenhagen DKK last close: 272.25


## 4. Financial line items from the 20-F

Source: Novo Nordisk Form 20-F for FY 2025, filed with the SEC on 4 February 2026. Line items below are in DKK millions, both for FY 2025 (year ended 31 December 2025) and FY 2024 (year ended 31 December 2024). The 20-F exhibit was tabulated using the consolidated financial statements; values were cross-checked against the company's annual report and a third-party aggregation.

FY 2024 figures are required for the F-Score year-over-year comparators (ROA, leverage, current ratio, gross margin, asset turnover).

In [7]:
# Source: Novo Nordisk Form 20-F for FY 2025, filed with the SEC on 4 February 2026.
# Filing URL: https://www.novonordisk.com/content/dam/nncorp/global/en/investors/irmaterial/annual_report/2026/novo-nordisk-form-20-f-2025.pdf
# Values were extracted via stockanalysis.com (which aggregates SEC EDGAR XBRL data) and
# cross-referenced against Novo Nordisk's Q4 2025 / FY 2025 press release headline figures
# (sales DKK 309.1 billion, FY EPS 23.03 DKK). Before final report submission, each line item
# below should be reconciled directly against the 20-F PDF on the indicated section.
#
# Section references in the consolidated financial statements of the 20-F:
#   - Revenue, COGS, Net Income      -> Consolidated Income Statement
#   - Total Assets, Current Assets,
#     Total Liabilities, Current Liabilities,
#     Long-Term Borrowings, Total Equity -> Consolidated Balance Sheet
#   - CFO, LT Debt Issued/Repaid,
#     Equity Issued/Repurchased       -> Consolidated Statement of Cash Flows (financing section)
#
# All values in DKK millions unless flagged otherwise.
fin = {
    'revenue':              {'2025': 309064, '2024': 290403},
    'cogs':                 {'2025':  58788, '2024':  44522},
    'gross_profit':         {'2025': 250276, '2024': 245881},
    'net_income':           {'2025': 102434, '2024': 100988},
    'cfo':                  {'2025': 119102, '2024': 120968},
    'total_assets':         {'2025': 542902, '2024': 465630},
    'total_current_assets': {'2025': 172453, '2024': 161788},
    'total_liabilities':    {'2025': 348855, '2024': 322144},
    'total_current_liab':   {'2025': 215661, '2024': 217614},
    'long_term_debt':       {'2025': 118941, '2024':  89674},
    'total_equity':         {'2025': 194047, '2024': 143486},
    'lt_debt_issued':       {'2025': 103931, '2024':  79391},
    'lt_debt_repaid':       {'2025':  79188, '2024':   6335},
    'equity_issued':        {'2025':      0, '2024':      0},
    'shares_repurchased':   {'2025':   1388, '2024':  20181},
}

# Total shares outstanding at 31 December 2025.
# Source: 20-F share-capital disclosure. Total share capital DKK 451 million at par value
# DKK 0.10 per share = 4.510 billion shares issued (1.075 billion A-shares + 3.435 billion
# B-shares). Less ~62 million treasury B-shares held by the company at year-end gives
# approximately 4.448 billion outstanding. This figure is used as the denominator for both
# Insider Selling and the Book-to-Market market capitalisation.
TOTAL_SHARES_OUTSTANDING = 4_448_000_000

pd.DataFrame(fin).T

,2025,2024
revenue,309064,290403
cogs,58788,44522
gross_profit,250276,245881
net_income,102434,100988
cfo,119102,120968
total_assets,542902,465630
total_current_assets,172453,161788
total_liabilities,348855,322144
total_current_liab,215661,217614
long_term_debt,118941,89674


## 5. Insider transactions (Article 19 PDMR)

As a Danish foreign private issuer, NVO is exempt from Section 16, so no Form 4 filings exist. Insider transactions are reported under Article 19 of EU MAR (Regulation 596/2014), with notifications published as company announcements on Globenewswire and on the Danish Financial Supervisory Authority's OAM database.

All NVO Article 19 PDMR notifications between 1 February 2026 and 30 April 2026 were reviewed. Only one notification falls in the trailing 3-month window ending 30 April 2026 and involves a CEO, CFO, COO, or Director: a sale by CFO Karsten Munk Knudsen on 10 February 2026 (announced 11 February 2026).

In [8]:
# Source: Globenewswire, NVO company announcement dated 11 February 2026.
insider_transactions = pd.DataFrame([
    {'date': '2026-02-10', 'role': 'CFO', 'name': 'Karsten Munk Knudsen',
     'instrument': 'NVO B-shares', 'nature': 'Sale', 'shares': 26246, 'price_dkk': 315.60,
     'venue': 'JANE STREET NETHERLANDS B.V.'},
    {'date': '2026-02-10', 'role': 'CFO', 'name': 'Karsten Munk Knudsen',
     'instrument': 'NVO B-shares', 'nature': 'Sale', 'shares':   311, 'price_dkk': 315.70,
     'venue': 'CBOE EUROPE - DXE PERIODIC (NL)'},
])
shares_sold_open_market = int(insider_transactions['shares'].sum())
shares_purchased_open_market = 0
print(insider_transactions)
print(f"\nNet shares sold by CEO/CFO/COO/Directors: {shares_sold_open_market - shares_purchased_open_market}")

         date role                  name    instrument nature  shares  \
0  2026-02-10  CFO  Karsten Munk Knudsen  NVO B-shares   Sale   26246   
1  2026-02-10  CFO  Karsten Munk Knudsen  NVO B-shares   Sale     311   

   price_dkk                            venue  
0      315.6     JANE STREET NETHERLANDS B.V.  
1      315.7  CBOE EUROPE - DXE PERIODIC (NL)  

Net shares sold by CEO/CFO/COO/Directors: 26557


## 6. Signal computations

All ten signals from the Part 1 specification. F-Score is integer (0-9); the rest are decimal.

In [9]:
# 6.1 Insider Selling
inssell = (shares_sold_open_market - shares_purchased_open_market) / TOTAL_SHARES_OUTSTANDING
print(f"Insider Selling: {inssell:.10f}")

Insider Selling: 0.0000059705


In [10]:
# 6.2 Book-to-Market (DKK basis)
novob_close_cutoff = float(novob.loc[novob.index == CUTOFF, 'Adj Close'].iloc[0])
market_cap_dkk = novob_close_cutoff * TOTAL_SHARES_OUTSTANDING
btom = (fin['total_equity']['2025'] * 1e6) / market_cap_dkk
print(f"NOVO-B close on {CUTOFF.date()}: DKK {novob_close_cutoff:.2f}")
print(f"Market cap: DKK {market_cap_dkk/1e9:.0f} bn")
print(f"Book equity: DKK {fin['total_equity']['2025']/1e3:.0f} bn")
print(f"Book-to-Market: {btom:.4f}")

NOVO-B close on 2026-04-30: DKK 272.25
Market cap: DKK 1211 bn
Book equity: DKK 194 bn
Book-to-Market: 0.1602


In [11]:
# 6.3 Momentum (USD/ADR basis with S&P 500 as the market benchmark)
def closest_close(df, target):
    sub = df[df.index <= target]
    return float(sub['Adj Close'].iloc[-1])

nvo_end = closest_close(nvo, MOMENTUM_WINDOW_END)
nvo_start = closest_close(nvo, MOMENTUM_WINDOW_START)
spx_end = closest_close(spx, MOMENTUM_WINDOW_END)
spx_start = closest_close(spx, MOMENTUM_WINDOW_START)

nvo_ret = nvo_end / nvo_start - 1
spx_ret = spx_end / spx_start - 1
momentum = nvo_ret - spx_ret

print(f"NVO ADR  cum return ({MOMENTUM_WINDOW_START.date()} -> {MOMENTUM_WINDOW_END.date()}): {nvo_ret:.4f}")
print(f"S&P 500 cum return: {spx_ret:.4f}")
print(f"Momentum (market-adjusted): {momentum:.4f}")

NVO ADR  cum return (2025-03-31 -> 2026-03-31): -0.4452
S&P 500 cum return: 0.1633
Momentum (market-adjusted): -0.6085


In [12]:
# 6.4 Gross Profitability
grossprofit = (fin['revenue']['2025'] - fin['cogs']['2025']) / fin['total_assets']['2025']
print(f"Gross Profitability: {grossprofit:.4f}")

Gross Profitability: 0.4610


In [13]:
# 6.5 Earnings Surprise (DKK basis: IBES EPS, NOVO-B price)
novob_5d_before = float(novob.loc[novob.index < Q4_2025_ANNOUNCEMENT, 'Adj Close'].iloc[-5])
novob_5d_date = novob.loc[novob.index < Q4_2025_ANNOUNCEMENT].index[-5]
earnsurprise = (Q4_2025_ACTUAL_DKK - Q4_2025_CONSENSUS_DKK) / novob_5d_before
print(f"NOVO-B price 5 trading days before {Q4_2025_ANNOUNCEMENT.date()}: DKK {novob_5d_before:.4f} (on {novob_5d_date.date()})")
print(f"Q4 2025 actual - consensus: {Q4_2025_ACTUAL_DKK - Q4_2025_CONSENSUS_DKK:.4f} DKK")
print(f"Earnings Surprise: {earnsurprise:.4f}")

NOVO-B price 5 trading days before 2026-02-03: DKK 380.8156 (on 2026-01-27)
Q4 2025 actual - consensus: -0.3000 DKK
Earnings Surprise: -0.0008


In [14]:
# 6.6 SUE
sue = (Q4_2025_ACTUAL_DKK - Q4_2024_ACTUAL_DKK) / novob_5d_before
print(f"Q4 2025 actual - Q4 2024 actual: {Q4_2025_ACTUAL_DKK - Q4_2024_ACTUAL_DKK:.4f} DKK")
print(f"SUE: {sue:.4f}")

Q4 2025 actual - Q4 2024 actual: -0.3000 DKK
SUE: -0.0008


In [15]:
# 6.7 F-Score (Piotroski 9-criterion)
checks = {}
checks['ROA > 0'] = int(fin['net_income']['2025'] / fin['total_assets']['2025'] > 0)
checks['CFO > 0'] = int(fin['cfo']['2025'] > 0)
checks['DROA > 0'] = int(
    fin['net_income']['2025'] / fin['total_assets']['2025']
    > fin['net_income']['2024'] / fin['total_assets']['2024']
)
checks['CFO > NI'] = int(fin['cfo']['2025'] > fin['net_income']['2025'])
checks['DLeverage < 0'] = int(
    fin['long_term_debt']['2025'] / fin['total_assets']['2025']
    < fin['long_term_debt']['2024'] / fin['total_assets']['2024']
)
checks['DCurrent ratio > 0'] = int(
    fin['total_current_assets']['2025'] / fin['total_current_liab']['2025']
    > fin['total_current_assets']['2024'] / fin['total_current_liab']['2024']
)
checks['No equity issuance'] = int(fin['equity_issued']['2025'] == 0)
checks['DGross margin > 0'] = int(
    (fin['revenue']['2025'] - fin['cogs']['2025']) / fin['revenue']['2025']
    > (fin['revenue']['2024'] - fin['cogs']['2024']) / fin['revenue']['2024']
)
checks['DAsset turnover > 0'] = int(
    fin['revenue']['2025'] / fin['total_assets']['2025']
    > fin['revenue']['2024'] / fin['total_assets']['2024']
)
fscore = sum(checks.values())
for k, v in checks.items():
    print(f"  {k:25s} {v}")
print(f"F-Score: {fscore}")

  ROA > 0                   1
  CFO > 0                   1
  DROA > 0                  0
  CFO > NI                  1
  DLeverage < 0             0
  DCurrent ratio > 0        1
  No equity issuance        1
  DGross margin > 0         0
  DAsset turnover > 0       0
F-Score: 5


In [16]:
# 6.8 External Financing
xfin_num = (
    fin['lt_debt_issued']['2025'] + fin['equity_issued']['2025']
    - fin['lt_debt_repaid']['2025'] - fin['shares_repurchased']['2025']
)
xfin = xfin_num / fin['total_assets']['2025']
print(f"Net financing (DKK m): {xfin_num:,}")
print(f"External Financing: {xfin:.4f}")

Net financing (DKK m): 23,355
External Financing: 0.0430


In [17]:
# 6.9 Volatility (NVO ADR, simple daily returns, 12 months ending 30 Apr 2026)
vol_window = nvo[(nvo.index > VOL_WINDOW_START) & (nvo.index <= VOL_WINDOW_END)]
daily_returns = vol_window['Adj Close'].pct_change().dropna()
volatility = float(daily_returns.std())
print(f"N daily returns: {len(daily_returns)}")
print(f"Volatility: {volatility:.4f}")

N daily returns: 250
Volatility: 0.0333


In [18]:
# 6.10 Accruals (FY annual)
accruals = (fin['net_income']['2025'] - fin['cfo']['2025']) / fin['total_assets']['2025']
print(f"Accruals: {accruals:.4f}")

Accruals: -0.0307


## 7. Output table

Columns: Signal | Value | Source | As-Of.

In [19]:
table = pd.DataFrame([
    {'Signal': 'Insider Selling',     'Value': inssell,
     'Source': 'Article 19 PDMR notifications (Globenewswire / Danish FSA OAM)',
     'As-Of': '2026-02-10'},
    {'Signal': 'Book-to-Market',      'Value': btom,
     'Source': '20-F FY 12/31/25 (book equity); Yahoo! Finance NOVO-B close (market cap)',
     'As-Of': '2026-04-30'},
    {'Signal': 'Momentum',            'Value': momentum,
     'Source': 'Yahoo! Finance (NVO ADR; S&P 500 benchmark)',
     'As-Of': '2026-03-31'},
    {'Signal': 'Gross Profitability', 'Value': grossprofit,
     'Source': '20-F FY 12/31/25',
     'As-Of': '2025-12-31'},
    {'Signal': 'Earnings Surprise',   'Value': earnsurprise,
     'Source': '6-K filed 2/3/26; WRDS IBES (Q1-Q3 2025 actuals; FY consensus); Yahoo! Finance NOVO-B price',
     'As-Of': '2026-02-03'},
    {'Signal': 'SUE',                 'Value': sue,
     'Source': '6-K filed 2/3/26; WRDS IBES (Q1-Q3 2025 actuals; Q4 2024 actual); Yahoo! Finance NOVO-B price',
     'As-Of': '2026-02-03'},
    {'Signal': 'F-Score',             'Value': fscore,
     'Source': '20-F FY 12/31/25 vs. 12/31/24',
     'As-Of': '2025-12-31'},
    {'Signal': 'External Financing',  'Value': xfin,
     'Source': '20-F FY 12/31/25 (financing activities)',
     'As-Of': '2025-12-31'},
    {'Signal': 'Volatility',          'Value': volatility,
     'Source': 'Yahoo! Finance (NVO ADR daily adjusted close)',
     'As-Of': '2026-04-30'},
    {'Signal': 'Accruals',            'Value': accruals,
     'Source': '20-F FY 12/31/25',
     'As-Of': '2025-12-31'},
])

def fmt(row):
    if row['Signal'] == 'F-Score':
        return int(row['Value'])
    if row['Signal'] == 'Insider Selling':
        return f"{row['Value']:.7f}"
    return f"{row['Value']:.4f}"

out = table.copy()
out['Value'] = table.apply(fmt, axis=1)
print(out.to_string(index=False))

out.to_csv(OUTPUT_DIR / 'nvo_signal_table.csv', index=False)
print('Wrote outputs/nvo_signal_table.csv')

             Signal     Value                                                                                        Source      As-Of
    Insider Selling 0.0000060                                Article 19 PDMR notifications (Globenewswire / Danish FSA OAM) 2026-02-10
     Book-to-Market    0.1602                      20-F FY 12/31/25 (book equity); Yahoo! Finance NOVO-B close (market cap) 2026-04-30
           Momentum   -0.6085                                                   Yahoo! Finance (NVO ADR; S&P 500 benchmark) 2026-03-31
Gross Profitability    0.4610                                                                              20-F FY 12/31/25 2025-12-31
  Earnings Surprise   -0.0008   6-K filed 2/3/26; WRDS IBES (Q1-Q3 2025 actuals; FY consensus); Yahoo! Finance NOVO-B price 2026-02-03
                SUE   -0.0008 6-K filed 2/3/26; WRDS IBES (Q1-Q3 2025 actuals; Q4 2024 actual); Yahoo! Finance NOVO-B price 2026-02-03
            F-Score         5                          

## 8. Notes on signal computation

Novo Nordisk is a Danish foreign private issuer, so several signal sources differ from a domestic US issuer's:

1. **Annual report.** 20-F is the relevant SEC filing. Financials are reported in DKK under IFRS. Line-item mapping to the Part 1 definitions is identical except where flagged.
2. **Quarterly disclosure.** NVO files Form 6-K for the Q4 2025 earnings release. The 6-K filing date (3 February 2026) is used as the announcement date for Earnings Surprise and SUE.
3. **Insider transactions.** Section 16 does not apply to foreign private issuers, so no Form 4 filings exist. Article 19 PDMR notifications under EU MAR were used instead, sourced from Globenewswire and the Danish FSA OAM database. Within the trailing 3-month window ending 30 April 2026, only one notification involves a CEO, CFO, COO, or Director: a 26,557 B-share open-market sale by CFO Karsten Munk Knudsen on 10 February 2026.
4. **EPS basis for Earnings Surprise / SUE.** For an IFRS DKK reporter, the internally consistent calculation uses DKK throughout: IFRS DKK quarterly EPS from WRDS IBES and the press release, divided by NOVO-B Copenhagen DKK price. WRDS IBES had not yet ingested NVO Q4 2025 actual at the time of the pull, so Q4 2025 EPS was recovered as `FY 2025 reported (23.03 DKK) − sum(Q1-Q3 2025 IBES actuals)`. Q4 2025 implied consensus was derived analogously from the latest pre-announcement annual consensus (statpers 15 January 2026, 23.33 DKK).
5. **Price universe for Momentum and Volatility.** NVO ADR daily adjusted close (USD) was used. The S&P 500 was the market benchmark for the market-adjusted return.
6. **Book-to-Market.** Numerator (book equity) and denominator (market cap) are both in DKK, with the NOVO-B Copenhagen close on 30 April 2026 used for market cap.
7. **External Financing.** NVO's 2025 financing activity is dominated by debt issuance, partly to fund the December 2025 Akero Therapeutics acquisition. Repurchase of own shares is also non-zero. The net result is a positive External Financing value.
8. **Accruals.** Computed on an FY annual basis: net income less cash flow from operations, scaled by total assets at fiscal year end.